## Cargar Archivos

In [8]:
import os
import re

folder = os.path.join(os.getcwd(), "genes")

# Carga cada archivo de genes/ en una variable con el mismo nombre del archivo
for file in sorted(os.listdir(folder)):
    if file.endswith(".txt"):
        name = re.sub(r"\W", "_", os.path.splitext(file)[0])
        with open(os.path.join(folder, file), encoding="utf-8") as f:
            next(f)  # salta la primera línea (encabezado)
            globals()[name] = "".join(f.read().split())  # sin saltos de línea

gen_M, gen_ORF1AB, gen_S, seq_proteins, SARS_COV_2_MN908947_3, SARS_COV_2_MT106054_1

('...>QHD43415_2AYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGG>QHD43415_3APTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGI

## KMP CON LPS PARA STRING MATCHING

In [9]:
def construir_lps(patron):
    """Tabla de prefijo-sufijo más largo (failure function) del patrón."""
    lps = [0] * len(patron)
    longitud = 0
    i = 1
    while i < len(patron):
        if patron[i] == patron[longitud]:
            longitud += 1
            lps[i] = longitud
            i += 1
        elif longitud:
            longitud = lps[longitud - 1]
        else:
            lps[i] = 0
            i += 1
    return lps


def kmp_buscar(texto, patron):
    """Devuelve los índices (base 0) donde aparece patron en texto."""
    if not patron:
        return []
    lps = construir_lps(patron)
    indices = []
    j = 0
    for i, c in enumerate(texto):
        while j and c != patron[j]:
            j = lps[j - 1]
        if c == patron[j]:
            j += 1
        if j == len(patron):
            indices.append(i - j + 1)
            j = lps[j - 1]
    return indices

In [10]:
# Punto 1: apariciones de cada gen en el genoma MN908947.3 (índices base 0)
genes = {"M": gen_M, "S": gen_S, "ORF1AB": gen_ORF1AB}

for nombre, gen in genes.items():
    indices = kmp_buscar(SARS_COV_2_MN908947_3, gen)
    print(f"Gen {nombre}")
    print(f"  Índices de aparición: {indices}")
    print(f"  Primeros 12 caracteres: {gen[:12]}")

Gen M
  Índices de aparición: [26522]
  Primeros 12 caracteres: ATGGCAGATTCC
Gen S
  Índices de aparición: [21562]
  Primeros 12 caracteres: ATGTTTGTTTTT
Gen ORF1AB
  Índices de aparición: [265]
  Primeros 12 caracteres: ATGGAGAGCCTT


In [11]:
def manacher(texto):
    """Devuelve (inicio, longitud) del palíndromo más largo de texto, en O(n)."""
    # Separadores '#' unifican palíndromos de longitud par e impar
    t = "^#" + "#".join(texto) + "#$"
    radios = [0] * len(t)
    centro = derecha = 0
    for i in range(1, len(t) - 1):
        if i < derecha:
            radios[i] = min(derecha - i, radios[2 * centro - i])
        while t[i + radios[i] + 1] == t[i - radios[i] - 1]:
            radios[i] += 1
        if i + radios[i] > derecha:
            centro, derecha = i, i + radios[i]
    longitud, centro = max((r, i) for i, r in enumerate(radios))
    inicio = (centro - longitud) // 2
    return inicio, longitud

In [12]:
# Punto 2: palíndromo más largo de cada gen (se guarda en palindromos.txt)
with open("palindromos.txt", "w", encoding="utf-8") as salida:
    for nombre, gen in genes.items():
        inicio, longitud = manacher(gen)
        palindromo = gen[inicio:inicio + longitud]
        print(f"Gen {nombre}: longitud {longitud} (índice {inicio}) -> {palindromo}")
        salida.write(f"Gen {nombre}\nLongitud: {longitud}\nÍndice: {inicio}\nPalíndromo: {palindromo}\n\n")

Gen M: longitud 11 (índice 493) -> CTAAAGAAATC
Gen S: longitud 15 (índice 2386) -> ATTTTGGTGGTTTTA
Gen ORF1AB: longitud 20 (índice 9711) -> CTCAATGACTTCAGTAACTC
